# [Chapter 8 — Parameter Estimation, §8.3] The Susceptible-Viewpoint Estimator $\hat\lambda = J/S^*$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 8 — Parameter Estimation
**Considerations developed:** 4, 5, 8
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
The conventional susceptible-viewpoint estimator $\hat\lambda = J/S^*$ requires the modeler to specify $S^*$. Demonstrates that the estimate is correct when $S^*$ is correct, and biased proportionally when $S^*$ is wrong.

## What you should already know
Chapter 8 alpha-estimator notebook.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import (
    book_style,
    BOOK_COLORS,
    integrate_sir_i,
    baseline_central_comparison,
)
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance

set_seed_chapter_08()
book_style()


## The estimator

If we know (or assume) the susceptible population $S^*$, the susceptible-viewpoint estimator is:

$$\hat\lambda(t) = \frac{J(t)}{S^*}$$

The true value is $\lambda(t) = c_S \beta (I(t)/N)$. When $S^*$ matches the true $S(t)$, this gives an unbiased estimate of $\lambda$.

In [ ]:
params = baseline_chapter_08()
true_R0 = params['c_I'] * params['beta'] * params['tau_R']
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']

alpha_true = params['c_I'] * params['beta'] * (S / params['N'])
J_true = alpha_true * I
J_obs = np.maximum(J_true + params['noise_sigma'] * J_true.max() * np.random.randn(len(t)), 0)

# Three S* assumptions: too low, correct, too high
S_star_assumptions = [0.7 * params['N'], 1.0 * params['N'], 1.3 * params['N']]
labels = ['S* = 0.7N (too low)', 'S* = N (matches early outbreak)', 'S* = 1.3N (too high)']

print(f"True early-outbreak lambda = c_S * beta * (I_avg/N)")
mask = (t > 5) & (t < 25)
lambda_true_early = params['c_S'] * params['beta'] * (I[mask].mean() / params['N'])
print(f"  ≈ {lambda_true_early:.6f}\n")

for S_star, label in zip(S_star_assumptions, labels):
    lambda_hat = J_obs[mask].mean() / S_star
    R0_est = lambda_hat * params['tau_R'] / (I[mask].mean() / params['N'])
    print(f"  {label}:")
    print(f"    lambda_hat = {lambda_hat:.6f}")
    print(f"    R_0 estimate = {R0_est:.3f}  (true = {true_R0:.2f})")


## Sensitivity to $S^*$

Sweep $S^*$ over a range and plot the resulting $\mathcal{R}_0$ estimate. The susceptible-viewpoint estimator's $\mathcal{R}_0$ is inversely proportional to $S^*$ — unit sensitivity.

In [ ]:
S_star_range = np.linspace(0.5, 1.5, 21) * params['N']
mask = (t > 5) & (t < 25)
J_avg = J_obs[mask].mean()
I_avg_over_N = I[mask].mean() / params['N']

R0_estimates_lambda = (J_avg / S_star_range) * params['tau_R'] / I_avg_over_N

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(S_star_range / params['N'], R0_estimates_lambda, color=BOOK_COLORS['lambda_hat'], lw=2,
        label=r'$\hat{R}_0$ from $\hat\lambda$')
ax.axhline(true_R0, ls='--', color=BOOK_COLORS['neutral'], lw=1, alpha=0.7, label=f'True R_0 = {true_R0:.1f}')
ax.axvline(1.0, ls=':', color=BOOK_COLORS['neutral'], lw=1, alpha=0.5, label='Correct S*/N = 1')
ax.set_xlabel('Assumed $S^*/N$')
ax.set_ylabel(r'$\hat{R}_0$ estimate')
ax.set_title(r'Susceptible-viewpoint $\hat\lambda$: unit sensitivity to $S^*$')
ax.legend()
plt.tight_layout()
plt.show()

print('Sensitivity index of R_0_estimate to S*: -1 (inversely proportional).')
print('When S* is wrong by 30%, R_0 is wrong by 30%.')
